# Shark Knowledge Medallion — Composición y Transición de Cada Capa

En este notebook exploramos en detalle la composición de cada capa de la arquitectura medallón y cómo los datos transicionan entre ellas.

---

## 1. Panorama general de la arquitectura medallón

La arquitectura medallón organiza los datos en cinco etapas progresivas:

```
                       ┌─────────────────────┐
                       │       BRONZE        │
                       │     (JSON crudo)    │
                       └────────┬────────────┘
                                │
                                ▼
                       ┌─────────────────────┐
                       │       SILVER        │
                       │     (validación)    │
                       └────────┬────────────┘
                                │
                                ▼
                       ┌─────────────────────┐
                       │      MERGE          │
                       │     (integración)   │
                       └────────┬────────────┘
                                │
                                ▼
                       ┌─────────────────────┐
                       │      WAREHOUSE      │
                       │      (DuckDB)       │
                       └────────┬────────────┘
                                │
                                ▼
                       ┌────────────────────┐
                       │       GOLD         │
                       │   (conocimiento)   │
                       └────────────────────┘
```

### Principios clave

- **Determinismo**: mismo input → mismo output en cada corrida.
- **Idempotencia**: reprocesar un lote produce filas_nuevas = 0.
- **Pivote Wikipedia**: si no hay página válida, la especie no avanza.
- **GBIF como augmentation**: taxonomía (solo EXACT) + ocurrencias agregadas.
- **Clave natural**: `species_key = slugify(scientific_name)`.

In [ ]:
import sys
sys.path.append("..")

In [ ]:
from src.pipeline import run_pipeline
from src.config.seed_species import SEED_SPECIES
from src.resolver.slugify import slugify
from src.bronze.fetcher import (
    fetch_gbif_taxonomy, fetch_gbif_occurrences,
    fetch_wikipedia_summary, fetch_wikipedia_extracts, persist_bronze,
)
from src.resolver.wikipedia import resolve_wikipedia_title, ResolutionStatus
from src.silver.models import WikipediaPage, GBIFTaxonomy, GBIFOccurrence
from src.silver.validator import (
    validate_wikipedia, validate_gbif_taxonomy, validate_gbif_occurrence, RejectionReason,
)
from src.merge.merger import merge_species
from src.warehouse.loader import Warehouse
from src.gold.models import Species, SpeciesChunk, SpeciesMetadata
from src.gold.indexer import build_species_from_merged, chunk_species, search
import json

In [ ]:
print(f'Lote semilla: {len(SEED_SPECIES)} especies')
print()
print('Ejemplo de species_key (clave natural que atraviesa todas las capas):')
for nombre in ['Carcharodon carcharias', 'Rhincodon typus', 'Galeocerdo cuvier']:
    print(f'  {nombre:30s} -> {slugify(nombre)}')

---
## 2. Bronze Layer — Raw JSON

### Propósito

Capturar la respuesta **íntegra** de cada API en archivos JSON con timestamp, sin transformar ni filtrar. Incluso errores HTTP se persisten tal cual.

### Estructura de salida

```
data/bronze/
├── gbif_taxonomy/lote_1__2026-07-22T10-00-00Z.json
├── gbif_occurrence/lote_1__2026-07-22T10-02-00Z.json
└── wikipedia/lote_1__2026-07-22T10-01-00Z.json
```

Cada archivo usa un **envelope de metadatos**:

```json
{
  "source": "wikipedia",
  "lote": 1,
  "ingested_at": "2026-07-22T10:00:00Z",
  "species_queried": [...],
  "raw_responses": [{"query": "...", "response": {...}}]
}
```

### 2.1 Query a GBIF Species Match — taxonomía

Endpoint: `GET https://api.gbif.org/v1/species/match?name={scientificName}`

In [ ]:
# Caso válido: Carcharodon carcharias -> match EXACT
tax_raw = fetch_gbif_taxonomy('Carcharodon carcharias')
print('>>> GBIF Species Match — Carcharodon carcharias')
print(f'HTTP {tax_raw["status"]}')
print(json.dumps(tax_raw['data'], indent=2))

In [ ]:
# Caso inválido: Fakeus nonexistus -> NO_MATCH (será rechazado en Silver)
tax_bad = fetch_gbif_taxonomy('Fakeus nonexistus')
print('>>> GBIF Species Match — Fakeus nonexistus (sin match exacto)')
print(f'HTTP {tax_bad["status"]}')
if isinstance(tax_bad['data'], dict):
    print(json.dumps(tax_bad['data'], indent=2))
else:
    print(tax_bad['data'])

### 2.2 Query a GBIF Occurrence Search — ocurrencias

Endpoint: `GET https://api.gbif.org/v1/occurrence/search?scientificName={nombre}&hasCoordinate=true&limit=50`

In [ ]:
# Caso válido: ocurrencias de Rhincodon typus (tiburón ballena)
occ_raw = fetch_gbif_occurrences('Rhincodon typus')
results = occ_raw['data'].get('results', [])
print('>>> GBIF Occurrence Search — Rhincodon typus')
print(f'HTTP {occ_raw["status"]}  |  Registros devueltos: {len(results)}\n')
for r in results[:3]:
    print(json.dumps({
        'gbifID': r.get('gbifID'),
        'country': r.get('country'),
        'decimalLatitude': r.get('decimalLatitude'),
        'decimalLongitude': r.get('decimalLongitude'),
        'eventDate': r.get('eventDate'),
        'basisOfRecord': r.get('basisOfRecord'),
        'iucnRedListCategory': r.get('iucnRedListCategory'),
    }, indent=2))
    print()

In [ ]:
# Otra especie: Carcharodon carcharias
occ_raw2 = fetch_gbif_occurrences('Carcharodon carcharias')
results2 = occ_raw2['data'].get('results', [])
print('>>> GBIF Occurrence Search — Carcharodon carcharias')
print(f'HTTP {occ_raw2["status"]}  |  Registros devueltos: {len(results2)}\n')
print(f'  Primer país:    {results2[0].get("country")}')
print(f'  Primeras coord: {results2[0].get("decimalLatitude")}, {results2[0].get("decimalLongitude")}')
print(f'  IUCN:           {results2[0].get("iucnRedListCategory")}')

### 2.3 Query a Wikipedia — summary + extracts

Dos endpoints por especie:
- `GET /api/rest_v1/page/summary/{title}` — párrafo introductorio
- `GET /w/api.php?action=query&prop=extracts&explaintext=true&titles={title}` — secciones completas

In [ ]:
# Resolución de título: de nombre científico a título Wikipedia
print('>>> Resolución de títulos Wikipedia\n')
for nombre in ['Carcharodon carcharias', 'Great white shark', 'Fakeus nonexistus']:
    res = resolve_wikipedia_title(nombre)
    print(f'  Entrada: {nombre:30s} -> Título: {str(res.title):25s} | Status: {res.status.value}')

In [ ]:
# Resumen Wikipedia
wp_summary = fetch_wikipedia_summary('Great white shark')
print('>>> Wikipedia Summary — Great white shark')
print(f'HTTP {wp_summary["status"]}')
print(f'Título:  {wp_summary["data"].get("title")}')
print(f'URL:     {wp_summary["data"].get("content_urls", {}).get("desktop", {}).get("page", "")}')
print(f'Extracto:\n{wp_summary["data"].get("extract", "")[:600]}...')

In [ ]:
# Extractos completos para ver las secciones del artículo
wp_extracts = fetch_wikipedia_extracts('Great white shark')
pages = wp_extracts['data'].get('query', {}).get('pages', {})
for pid, pdata in pages.items():
    extract_text = pdata.get('extract', '')
    print(f'>>> Wikipedia Extracts — {pdata.get("title")}')
    print(f'Longitud total: {len(extract_text)} caracteres')
    print('Secciones detectadas:')
    for line in extract_text.split('\n'):
        if line.startswith('=='):
            print(f'  {line.strip()}')

### 2.4 Persistencia Bronze

Todo lo anterior se persiste como JSON crudo con envelope de metadatos. Nada se descarta.

In [ ]:
# Persistir una consulta real a Bronze
tax_demo = fetch_gbif_taxonomy('Galeocerdo cuvier')
path = persist_bronze(
    source='gbif_taxonomy',
    lote=2,
    species_queried=['Galeocerdo cuvier'],
    raw_responses=[{'query': 'Galeocerdo cuvier', 'response': tax_demo}],
)
print(f'Archivo creado: {path}\n')
with open(path) as f:
    envelope = json.load(f)
print(json.dumps(envelope, indent=2)[:600] + '...')

---
## 3. Silver Layer — Validación y contratos

### Propósito

Transformar el JSON crudo de Bronze en registros validados por **contratos Pydantic v2**. Cada fuente tiene su propio validador y su propia partición `valid/` / `rejected/`.

### Estructura de salida

```
data/silver/
├── wikipedia/valid/{species_key}.json
├── wikipedia/rejected/{batch_id}_wikipedia_{species_key}.json
├── gbif_taxonomy/valid/{species_key}.json
├── gbif_taxonomy/rejected/{batch_id}_gbif_taxonomy_{species_key}.json
├── gbif_occurrence/valid/{gbif_id}.json
└── gbif_occurrence/rejected/{batch_id}_gbif_occurrence_{gbif_id}.json
```

### Contratos (Pydantic v2)

In [ ]:
# WikipediaPage: contrato pivote (1 por species_key)
print('WikipediaPage:')
for name, field in WikipediaPage.model_fields.items():
    print(f'  {name:20s} | {str(field.annotation):25s} | required={field.is_required()}')

In [ ]:
print('GBIFTaxonomy:')
for name, field in GBIFTaxonomy.model_fields.items():
    print(f'  {name:20s} | {str(field.annotation):25s} | required={field.is_required()}')

In [ ]:
print('GBIFOccurrence:')
for name, field in GBIFOccurrence.model_fields.items():
    print(f'  {name:20s} | {str(field.annotation):25s} | required={field.is_required()}')

### 3.1 Validación Wikipedia — válido vs rechazado

In [ ]:
# CASO VÁLIDO: Great white shark
res = resolve_wikipedia_title('Carcharodon carcharias')
wp_summary = fetch_wikipedia_summary(res.title)
wp_extracts = fetch_wikipedia_extracts(res.title)
wp_valid = validate_wikipedia(
    {'raw': {'summary': wp_summary, 'extracts': wp_extracts}, 'resolved_via': res.resolved_via.value},
    batch_id='demo', species_key='carcharodon_carcharias',
)
print('>>> Wikipedia VÁLIDO' if wp_valid else '>>> Wikipedia RECHAZADO')
if wp_valid:
    print(f'  species_key:  {wp_valid["species_key"]}')
    print(f'  title:        {wp_valid["title"]}')
    print(f'  resolved_via: {wp_valid["resolved_via"]}')
    print(f'  extract_len:  {len(wp_valid["extract"])} chars')
    print(f'  sections_len: {len(wp_valid["sections_text"])} chars')

In [ ]:
# CASO RECHAZADO: especie inexistente -> PAGE_NOT_FOUND
res_bad = resolve_wikipedia_title('Fakeus nonexistus')
print(f'Status resolución: {res_bad.status.value}')
print('(No se puede validar Wikipedia — la especie se cuarentena directamente)')

### 3.2 Validación GBIF Taxonomy — válido vs rechazado

In [ ]:
# CASO VÁLIDO: match EXACT
tax_valid = validate_gbif_taxonomy(
    {'raw': fetch_gbif_taxonomy('Carcharodon carcharias')},
    batch_id='demo', species_key='carcharodon_carcharias',
)
print('>>> GBIF Taxonomy VÁLIDO' if tax_valid else '>>> GBIF Taxonomy RECHAZADO')
if tax_valid:
    print(f'  usage_key:      {tax_valid["usage_key"]}')
    print(f'  canonical_name: {tax_valid["canonical_name"]}')
    print(f'  rank:           {tax_valid["rank"]}')
    print(f'  kingdom:        {tax_valid["kingdom"]}')
    print(f'  phylum:         {tax_valid["phylum"]}')
    print(f'  class_name:     {tax_valid["class_name"]}')
    print(f'  match_type:     {tax_valid["match_type"]}')

In [ ]:
# CASO RECHAZADO: sin match exacto
tax_rejected = validate_gbif_taxonomy(
    {'raw': fetch_gbif_taxonomy('Fakeus nonexistus')},
    batch_id='demo', species_key='fakeus_nonexistus',
)
print('>>> GBIF Taxonomy VÁLIDO' if tax_rejected else '>>> GBIF Taxonomy RECHAZADO')
print('(rechazado por MATCH_TYPE_NOT_EXACT — la especie sigue siendo válida en Gold vía Wikipedia)')

### 3.3 Validación GBIF Occurrence — válido vs rechazado

In [ ]:
# CASO VÁLIDO: ocurrencia con todos los campos requeridos
occ_raw = fetch_gbif_occurrences('Carcharodon carcharias')
valid_occs = []
rejected_occs = []
for rec in occ_raw['data'].get('results', []):
    r = validate_gbif_occurrence(rec, batch_id='demo', species_key='carcharodon_carcharias')
    if r:
        valid_occs.append(r)
    else:
        rejected_occs.append(rec.get('gbifID', '?'))
print('>>> Validación de ocurrencias — Carcharodon carcharias')
print(f'  Válidas:   {len(valid_occs)}')
print(f'  Rechazadas:{len(rejected_occs)}')
print(f'  Total:     {len(occ_raw["data"].get("results", []))}')
print()
if valid_occs:
    print('Ejemplo de ocurrencia válida:')
    print(valid_occs[0])

In [ ]:
# Mostrar todos los motivos de rechazo disponibles en Silver
print('Motivos de rechazo:')
for r in RejectionReason:
    print(f'  {r.value}')

---
## 4. Gold Layer — Conocimiento canónico y búsqueda

### Transición Silver → Gold

Entre Silver y Gold ocurren tres procesos intermedios:

1. **Merge** — Wikipedia se usa como pivote (1:1), GBIF Taxonomy se une si EXACT, Occurrence se agrega (N→1).
2. **Warehouse** — DuckDB embebido con 5 tablas, UPSERT idempotente por clave natural.
3. **Gold** — Se construyen las entidades Species, SpeciesChunk, SpeciesMetadata; se generan embeddings y el índice FAISS.

### Entidades Gold

- **Species (Parent)**: documento completo con 12 secciones canónicas.
- **SpeciesChunk (Child)**: fragmentos semánticos por sección, cada uno con embedding 384d.
- **SpeciesMetadata**: filtros estructurados (taxonomía, IUCN, países, años).

### 4.1 Merge — Integración Wikipedia + GBIF

Fusionamos los datos validados de las tres fuentes en un solo registro por especie.

In [ ]:
# Pipeline manual de Silver -> Merge para una especie
sk = 'carcharodon_carcharias'

# 1. Silver: validar Wikipedia
res = resolve_wikipedia_title('Carcharodon carcharias')
wp_summary = fetch_wikipedia_summary(res.title)
wp_extracts = fetch_wikipedia_extracts(res.title)
wp_valid = validate_wikipedia(
    {'raw': {'summary': wp_summary, 'extracts': wp_extracts},
     'resolved_via': res.resolved_via.value},
    batch_id='demo', species_key=sk,
)

# 2. Silver: validar GBIF Taxonomy
tax_valid = validate_gbif_taxonomy(
    {'raw': fetch_gbif_taxonomy('Carcharodon carcharias')},
    batch_id='demo', species_key=sk,
)

# 3. Silver: validar ocurrencias
occ_valid = []
for rec in fetch_gbif_occurrences('Carcharodon carcharias')['data'].get('results', []):
    r = validate_gbif_occurrence(rec, batch_id='demo', species_key=sk)
    if r:
        occ_valid.append(r)

# 4. Merge: fusionar fuentes
merged = merge_species(wp_valid, tax_valid, occ_valid)
if merged:
    print('>>> Merge completo para Carcharodon carcharias')
    print(json.dumps(merged.model_dump(), indent=2, default=str))

### 4.2 Warehouse — Carga en DuckDB

Los registros fusionados se cargan en DuckDB con UPSERT idempotente.

In [ ]:
# Inicializar Warehouse y cargar datos
wh = Warehouse()
wh.connect()
wh.initialize_schema()

# Cargar Wikipedia validada
wh.upsert_wikipedia([wp_valid])
# Cargar taxonomía validada
wh.upsert_gbif_taxonomy([tax_valid])
# Cargar ocurrencias validadas
wh.upsert_gbif_occurrences(occ_valid)
# Cargar merged
wh.upsert_species_merged([merged.model_dump()])
# Registrar en catálogo
wh.upsert_species_catalog([{
    'species_key': sk,
    'scientific_name': 'Carcharodon carcharias',
    'wikipedia_title': merged.wikipedia_title,
    'origin': 'SEED',
}], batch_id='demo_medallones')

print('>>> Evidence del Warehouse tras la carga:')
for e in wh.get_evidence():
    print(f'  {e["tabla"]:30s} | {e["count"]} filas')

In [ ]:
# Consultar species_merged directamente desde DuckDB
conn = wh.connect()
row = conn.execute(
    'SELECT species_key, wikipedia_title, taxonomy_status, n_occurrences_validas, iucn_red_list_category, augmentation_status, paises '
    'FROM species_merged WHERE species_key = ?', [sk]
).fetchone()
columns = ['species_key', 'wikipedia_title', 'taxonomy_status', 'n_occurrences_validas', 'iucn_red_list_category', 'augmentation_status', 'paises']
if row:
    print('>>> Consulta directa a species_merged:')
    for c, v in zip(columns, row):
        print(f'  {c:30s} | {v}')

### 4.3 Gold — Chunking semántico

A partir del merged construimos el modelo **Species** con 12 secciones canónicas y lo fragmentamos en **SpeciesChunks**.

Secciones canónicas:
Identity, Taxonomy, Morphology, Distribution, Habitat, Behavior, Diet, Reproduction, Conservation, Human Interaction, References, Multimedia.

In [ ]:
print('Species — 12 secciones canónicas:')
for name, field in Species.model_fields.items():
    print(f'  {name:20s} | {field.annotation}')

In [ ]:
# Construir Species desde merged y generar chunks
species_dict = build_species_from_merged(merged.model_dump())
chunks = chunk_species(species_dict)

print('>>> Species construido:')
for k, v in species_dict.items():
    if k != 'species_key':
        status = '✅' if v else '❌'
        print(f'  {status} {k:20s} | {"presente" if v else "vacío"}')

print(f'\n>>> Chunks generados: {len(chunks)}')
for ch in chunks:
    print(f'  {ch["chunk_id"]}  ({len(ch["text"])} chars)')

### 4.4 Búsqueda híbrida con filtros

La función `search()` implementa retrieval Parent-Child:
1. Embed query con `BAAI/bge-small-en-v1.5`
2. FAISS oversampling (`top_k * 5`)
3. Dedup por species_key (mejor score)
4. Filtros estructurados opcionales: taxonomía, IUCN, país

**Nota**: La búsqueda requiere un índice FAISS preconstruido (correr el pipeline completo). Estos ejemplos muestran la firma y el uso esperado con filtros híbridos.

In [ ]:
# Demostración de la firma de search() con filtros
print('Firma de search():')
import inspect
sig = inspect.signature(search)
for name, param in sig.parameters.items():
    default = f' = {param.default}' if param.default is not inspect.Parameter.empty else ''
    print(f'  {name}: {param.annotation}{default}')

In [ ]:
# Ejemplo de llamada a search (requiere índice FAISS pre-construido)
print('Ejemplos de consulta híbrida con filtros:\n')
print('>>> Sin filtros:')
print('  search(query="large coastal shark", top_k=3)')
print('>>> Con filtro taxonómico:')
print('  search(')
    print('      query="shark that eats plankton",')
    print('      top_k=3,')
    print('      filtro_taxonomia={"class_name": "Chondrichthyes"},')
    print('  )')
print('>>> Con filtro IUCN:')
print('  search(')
    print('      query="endangered shark species",')
    print('      top_k=5,')
    print('      filtro_iucn="ENDANGERED",')
    print('  )')
print('>>> Con filtro geográfico:')
print('  search(')
    print('      query="sharks found near Australia",')
    print('      top_k=3,')
    print('      filtro_pais="Australia",')
    print('  )')
print('>>> Múltiples filtros combinados (AND):')
print('  search(')
    print('      query="reef shark habitat",')
    print('      top_k=3,')
    print('      filtro_taxonomia={"class_name": "Chondrichthyes"},')
    print('      filtro_pais="Mexico",')
    print('  )')

### 4.5 SpeciesMetadata — Filtros estructurados

Cada especie tiene metadatos asociados que permiten filtrar resultados por taxonomía, estado de conservación o geografía.

In [ ]:
print('SpeciesMetadata:')
for name, field in SpeciesMetadata.model_fields.items():
    print(f'  {name:25s} | {field.annotation}')

---
## Resumen de la transición de datos

| Capa | Formato | Qué ocurre |
|------|---------|------------|
| **Bronze** | JSON crudo + envelope | Se persiste la respuesta exacta de cada API, sin transformación |
| **Silver** | JSON validado (Pydantic) | Se validan contratos; válidos van a `valid/`, inválidos a `rejected/` con motivo explícito |
| **Merge** | Registro fusionado | Wikipedia pivote 1:1 + GBIF Taxonomy (si EXACT) + Occurrence agregada (N→1) |
| **Warehouse** | DuckDB (5 tablas) | UPSERT idempotente por clave natural; `species_catalog` como registro maestro |
| **Gold** | FAISS + Parquet | Species (12 secciones), SpeciesChunks (embeddings 384d), SpeciesMetadata (filtros) + `search()` híbrido |

---
*Shark Knowledge Medallion — Composición y transición de datos a través de la arquitectura medallón*